In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
os.makedirs(raw_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py.")

# --- 2. Registry Lookup (TGM) ---
SOURCE_PREFIX = "TGM"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, config_dir=config_dir, 
    fallback_name="Thesaurus for Graphic Materials", 
    fallback_uri="http://id.loc.gov/vocabulary/graphicMaterials/"
)

SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})', 'Accept': 'application/json'}

# --- 3. Verified TGM Seed URIs ---
SEED_URIS = [
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008780", # Religious fundamentalism
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008767", # Religion
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm013160", # Reincarnation
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm004966", # Hell
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm004956", # Heaven
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm010400", # Supernatural practices
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008146", # Prayer
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm009319", # Seances
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008788", # Religious retreats
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm007802", # Pilgrimages
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008787", # Religious processions
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm000987", # Biblical events
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm013016", # Sacred space
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008321", # Prophets
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm007616", # People associated with religion
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008782", # Religious groups
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm008771", # Religious articles
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm013226", # Hope
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm004711", # Gratitude
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm004231", # Forgiveness
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm010680", # Ten commandments
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm010398", # Supernatural
    "http://id.loc.gov/vocabulary/graphicMaterials/tgm009869"  # Soul
]

# --- 4. Persistent Progress Tracking ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file)
    visited_uris = set(existing_df['URI'].unique().tolist())
    print(f"[*] Resuming: Found {len(visited_uris)} TGM concepts already saved.")
else:
    visited_uris = set()
    print("[*] Starting fresh TGM ingest.")

# --- 5. Bottom-Up Ancestry Resolution ---
ancestry_cache = {}

def get_full_tgm_path(uri, depth=0):
    """Recursively fetches broader terms to build an absolute hierarchy path from the root."""
    if not uri: return ""
    clean_uri = uri.replace('.json', '').replace('.html', '').rstrip('/')
    
    if clean_uri in ancestry_cache: return ancestry_cache[clean_uri]
    if depth > 10: return clean_uri.split('/')[-1] # Depth limit
        
    try:
        res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            label = clean_uri.split('/')[-1]
            for pref in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
                if pref.get('@language', 'en').startswith('en'): label = pref.get('@value', label)

            broader_nodes = node.get('http://www.w3.org/2004/02/skos/core#broader', [])
            if not broader_nodes:
                broader_nodes = node.get('http://www.loc.gov/mads/rdf/v1#hasBroaderAuthority', [])
            
            if broader_nodes and broader_nodes[0].get('@id'):
                parent_uri = broader_nodes[0].get('@id')
                parent_path = get_full_tgm_path(parent_uri, depth + 1)
                full_path = f"{parent_path} > {label}" if parent_path else label
                ancestry_cache[clean_uri] = full_path
                return full_path
            else:
                ancestry_cache[clean_uri] = label
                return label
    except Exception:
        pass
    return clean_uri.split('/')[-1]

# --- 6. Extraction Logic ---
def process_lc_node(uri):
    """Recursively crawls narrower concepts with persistent skipping."""
    clean_uri = uri.rstrip('/').replace('.json', '').replace('.rdf', '').replace('.html', '')
    
    if clean_uri in visited_uris:
        return
    visited_uris.add(clean_uri)
    
    time.sleep(0.8) # Polite delay for LoC
    
    node_data = None
    try:
        res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=15)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node_data = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
    except Exception: 
        pass
            
    if not node_data:
        return

    SKOS_PREF = 'http://www.w3.org/2004/02/skos/core#prefLabel'
    SKOS_ALT  = 'http://www.w3.org/2004/02/skos/core#altLabel'
    SKOS_BROADER = 'http://www.w3.org/2004/02/skos/core#broader'
    SKOS_NARROWER = 'http://www.w3.org/2004/02/skos/core#narrower'
    SKOS_SCOPE = 'http://www.w3.org/2004/02/skos/core#scopeNote'
    MADS_CITATION = 'http://www.loc.gov/mads/rdf/v1#citationNote'

    # 1. Label Extraction
    label = "No Label"
    has_translation = ""
    for pref in node_data.get(SKOS_PREF, []):
        lang = pref.get('@language', '').lower()
        if not lang or lang.startswith('en'): label = pref.get('@value', label)
        else: has_translation = "yes"
    
    print(f"\rIngesting [{len(visited_uris)}]: {label[:70]:<70}", end="", flush=True)

    # 2. Synonyms & Descriptions
    synonyms = [alt.get('@value') for alt in node_data.get(SKOS_ALT, []) if alt.get('@value')]
    descriptions = []
    for item in data:
        if isinstance(item, dict):
            for n in item.get(MADS_CITATION, []):
                if n.get('@value'): descriptions.append(n.get('@value'))
            for n in item.get(SKOS_SCOPE, []):
                if n.get('@value'): descriptions.append(n.get('@value'))
                
    # 3. True Ancestry and Parent Resolution
    broader_nodes = node_data.get(SKOS_BROADER, [])
    p_ids = [b.get('@id').split('/')[-1] for b in broader_nodes if b.get('@id')]
    absolute_path = get_full_tgm_path(clean_uri)
    
    cid = clean_uri.split('/')[-1]

    # 4. Map to 14-column schema
    extracted_data = {
        'Source_System': SOURCE_NAME, 
        'Primary_Label': label,
        'CURIE': f"{SOURCE_PREFIX}:{cid}", 
        'Formal_Label': label,
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': absolute_path, 
        'Synonyms': " | ".join(list(set(synonyms))),
        'Description': " | ".join(descriptions), 
        'Parent_IDs': " | ".join(p_ids),
        'Concept_ID': cid, 
        'URI': clean_uri, 
        'Has_Translation': has_translation,
        'Status': 'active'
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )

    # 5. Recursive Crawl Downward
    for child in node_data.get(SKOS_NARROWER, []):
        child_uri = child.get('@id')
        if child_uri and child_uri.startswith(BASE_URI):
            process_lc_node(child_uri)

# --- 7. Execution ---
print(f"Starting TGM Ingest for {len(SEED_URIS)} seeds...\n")

for seed in SEED_URIS:
    process_lc_node(seed)

print(f"\n\nIngest Complete! Total unique TGM concepts: {len(visited_uris)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[*] Resuming: Found 11 TGM concepts already saved.
Starting TGM Ingest for 23 seeds...

Ingesting [141]: Soul                                                                  

Ingest Complete! Total unique TGM concepts: 141


In [5]:
# ==============================================================================
# CELL 2: LATERAL DISCOVERY & SEED VALIDATION
# Run this cell manually to discover related concepts outside the current scope.
# ==============================================================================

import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = True  # Toggle to False once your TGM ingest is finalized

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup Paths & Environment ---
    load_dotenv("../../config/.env") 
    contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")
    
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
    raw_tgm_file = os.path.join(raw_data_dir, "raw_tgm.csv")
    output_candidates_file = os.path.join(raw_data_dir, "tgm_lateral_candidates.csv")
    
    HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})', 'Accept': 'application/json'}
    
    # --- 2. Load Existing Concepts (Deduplication Baseline) ---
    if not os.path.exists(raw_tgm_file):
        raise FileNotFoundError(f"Could not find {raw_tgm_file}. Please run Cell 1 first.")
        
    existing_df = pd.read_csv(raw_tgm_file)
    existing_uris = set(existing_df['URI'].astype(str).str.rstrip('/'))
    print(f"[*] Loaded {len(existing_uris)} existing TGM concepts to serve as the baseline.")
    
    # --- 3. Bottom-Up Ancestry Function (Duplicated for Cell Independence) ---
    ancestry_cache = {}
    def get_full_tgm_path(uri, depth=0):
        if not uri: return ""
        clean_uri = uri.replace('.json', '').replace('.html', '').rstrip('/')
        
        if clean_uri in ancestry_cache: return ancestry_cache[clean_uri]
        if depth > 10: return clean_uri.split('/')[-1]
            
        try:
            res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=10)
            if res.status_code == 200:
                data = res.json()
                if isinstance(data, dict): data = [data]
                
                node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
                label = clean_uri.split('/')[-1]
                
                for pref in node.get('http://www.w3.org/2004/02/skos/core#prefLabel', []):
                    if pref.get('@language', 'en').startswith('en'): label = pref.get('@value', label)
    
                broader_nodes = node.get('http://www.w3.org/2004/02/skos/core#broader', [])
                if not broader_nodes:
                    broader_nodes = node.get('http://www.loc.gov/mads/rdf/v1#hasBroaderAuthority', [])
                
                if broader_nodes and broader_nodes[0].get('@id'):
                    parent_uri = broader_nodes[0].get('@id')
                    parent_path = get_full_tgm_path(parent_uri, depth + 1)
                    full_path = f"{parent_path} > {label}" if parent_path else label
                    ancestry_cache[clean_uri] = full_path
                    return full_path
                else:
                    ancestry_cache[clean_uri] = label
                    return label
        except Exception:
            pass
        return clean_uri.split('/')[-1]

    # --- 4. Harvest Related Concepts ---
    candidates_data = []
    
    print("Scanning Library of Congress API for lateral associations...\n")
    for i, uri in enumerate(list(existing_uris), 1):
        clean_uri = uri.replace('.json', '').replace('.html', '')
        print(f"\r[{i}/{len(existing_uris)}] Checking relations for: {clean_uri.split('/')[-1]:<30}", end="", flush=True)
        
        try:
            res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=10)
            if res.status_code != 200: continue
                
            data = res.json()
            if isinstance(data, dict): data = [data]
            main_node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            related_nodes = main_node.get('http://www.w3.org/2004/02/skos/core#related', [])
            if not related_nodes:
                related_nodes = main_node.get('http://www.loc.gov/mads/rdf/v1#hasRelatedAuthority', [])
                
            for r_node in related_nodes:
                r_uri = r_node.get('@id', '').rstrip('/')
                
                # STEP 11: Deduplicate against captured data
                if not r_uri or r_uri in existing_uris: continue
                    
                # Prevent querying the exact same candidate multiple times in this run
                if not any(d['Candidate_URI'] == r_uri for d in candidates_data):
                    # Fetch the full path using our bottom-up crawler
                    candidate_path = get_full_tgm_path(r_uri)
                    candidate_label = candidate_path.split(' > ')[-1]
                    
                    candidates_data.append({
                        'Candidate_URI': r_uri, 
                        'Candidate_Label': candidate_label,
                        'Hierarchy_Path': candidate_path
                    })
        except Exception:
            pass
            
        time.sleep(0.5) # Be polite to LC servers
    
    # --- 5. Export and Sort ---
    print(f"\n\nLateral Discovery Complete! Found {len(candidates_data)} unique candidates outside current boundaries.")
    
    if candidates_data:
        candidates_df = pd.DataFrame(candidates_data)
        
        # Sort by hierarchy so entire branches group together visually in the CSV
        candidates_df = candidates_df.sort_values(by='Hierarchy_Path')
        candidates_df.to_csv(output_candidates_file, index=False, encoding='utf-8-sig')
        
        print(f"Sorted candidates saved to: {output_candidates_file}")
        print("\nNext Step: Open this CSV, review the Hierarchy_Path column, and identify any missing parent branches to add as seeds in Cell 1.")
    else:
        print("No new relevant concepts found. Your seed list is comprehensive!")
else:
    print("Lateral Discovery is disabled. Set RUN_LATERAL_DISCOVERY = True to execute.")

[*] Loaded 141 existing TGM concepts to serve as the baseline.
Scanning Library of Congress API for lateral associations...

[141/141] Checking relations for: tgm004956                     

Lateral Discovery Complete! Found 121 unique candidates outside current boundaries.
Sorted candidates saved to: c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\tgm_lateral_candidates.csv

Next Step: Open this CSV, review the Hierarchy_Path column, and identify any missing parent branches to add as seeds in Cell 1.
